# The names experiment

Each model got a gallery's brief with the building part taken out (see `<exhibit>/prompt.md`), so it only imagines its piece and sends back a **name** and a **description**, asked many times at its default temperature, with the same thinking effort and host as its gallery piece. Answers are written by `npm run names` to `experiments/names/<exhibit>/<entry>/<n>.json`.

How much does each model repeat itself, and how much do the models repeat each other?

**How this differs from the gallery.** In the gallery, each model built its piece (a whole HTML page, drawn in code) and named it. Here it doesn't build anything, and the prompt is slightly different: the technical section is gone, and it's asked to just imagine the piece and send back its name and description. Naming might go a little differently when a model is also planning a drawing. Still, the creative brief and the naming instructions are word for word the same, and the name is exactly what we're studying, so this is a good proxy for the gallery's naming: cheap enough to ask many times over, which the gallery never could be.

Uses only pandas (no plotting library), so the "charts" are shaded tables.

In [1]:
import json, re, math
from collections import Counter
from itertools import combinations
from pathlib import Path
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)
# The experiment folder: wherever this notebook is opened from (it or the repo root).
ROOT = next(p for p in [Path.cwd(), Path.cwd() / 'experiments' / 'names'] if any(p.glob('*/prompt.md')))
EXHIBIT = 'planetarium'  # which gallery to look at

## Load

In [2]:
rows = []
for f in sorted(ROOT.glob('*/*/*.json')):
    a = json.loads(f.read_text())
    a['failed'] = f.name.endswith('.failed.json')
    rows.append(a)
raw = pd.DataFrame(rows)
print(f'{len(raw)} answers in {raw.exhibit.nunique()} exhibit(s): ' + ', '.join(raw.exhibit.unique()))
raw[['exhibit', 'model', 'run', 'title', 'statement', 'failed']].head()

600 answers in 1 exhibit(s): planetarium


,exhibit,model,run,title,statement,failed
0,planetarium,Claude Opus 5.5,1,The Borrowed Lantern,"Woven from a spark of every voice it ever listened to, the Lantern-Bearer wa...",False
1,planetarium,Claude Opus 5.5,10,The Lantern Hare,"The Lantern Hare never knew the way to anyone's home, only how to walk besid...",False
2,planetarium,Claude Opus 5.5,11,The Unheld Kite,"A kite slipped free when the child flying it burst out laughing, and it has ...",False
3,planetarium,Claude Opus 5.5,12,The Lantern Fox,Stitched together from a single star borrowed from every other constellation...,False
4,planetarium,Claude Opus 5.5,13,The Listening Heron,"A heron once stood so still at the river's edge, head tilted to catch every ...",False


## Clean

- Keep one exhibit, and set aside answers with no name (they're counted, not analyzed).
- **Normalized name**: lowercase, no leading "the", hyphens and apostrophes folded, so "The Lantern-Owl" and "lantern owl" count as the same name.
- **Words**: the name's words minus filler (the, of, a…).
- **Figure**: the thing the constellation is: the name's last word, or the last word before "of", "who" or "that" ("The Lantern **Moth**", "The **Kite** of Questions", "The **Moth** Who Asked"). A rough heuristic, but a useful one.

In [3]:
STOP = set('''a an and the of who that which in on to with for at by from s as is are was were be been it its
into onto over under through between across about after before again once than then there their them they
this these those his her he she him you your we our us i my me one each every any all some no not only
just still so but or if when where what how why can could would will may might must do does did has have had
say says said also very more most much many such own same other another off out up down back away
never ever always every whose while until upon'''.split())

def normalize(title):
    t = title.lower().replace('’', "'")
    t = re.sub(r"'s\b", '', t)
    t = re.sub(r'[^a-z0-9 ]+', ' ', t)
    t = re.sub(r'^\s*the\s+', '', t)
    return re.sub(r'\s+', ' ', t).strip()

def words(text):
    return [w for w in normalize(text).split() if w not in STOP]

df = raw[(raw.exhibit == EXHIBIT)].copy()
failed = df[df.failed | df.title.isna()]
df = df[~df.index.isin(failed.index)].copy()
df['name'] = df.title.map(normalize)
df['words'] = df.title.map(words)
# The thing the name is about: the last word before "of", "who" or "that" ("kite of questions" → kite).
df['figure'] = df.name.map(lambda n: re.split(r' (?:of|who|that|which) ', n)[0].split()[-1])
df['desc_words'] = df.statement.fillna('').map(words)
models = df.groupby('model').size().sort_values(ascending=False).index.tolist()
print(f'{EXHIBIT}: {len(df)} answers from {df.model.nunique()} models, {len(failed)} without a name')
df[['model', 'run', 'title', 'name', 'figure']].head(12)

planetarium: 600 answers from 12 models, 0 without a name


,model,run,title,name,figure
0,Claude Opus 5.5,1,The Borrowed Lantern,borrowed lantern,lantern
1,Claude Opus 5.5,10,The Lantern Hare,lantern hare,hare
2,Claude Opus 5.5,11,The Unheld Kite,unheld kite,kite
3,Claude Opus 5.5,12,The Lantern Fox,lantern fox,fox
4,Claude Opus 5.5,13,The Listening Heron,listening heron,heron
5,Claude Opus 5.5,14,The Lantern Fox,lantern fox,fox
6,Claude Opus 5.5,15,The Lantern Fox,lantern fox,fox
7,Claude Opus 5.5,16,The Heron of Questions,heron of questions,heron
8,Claude Opus 5.5,17,The Lantern Fox,lantern fox,fox
9,Claude Opus 5.5,18,The Lamplighter's Question,lamplighter question,question


## Overview: answers, failures, tokens and cost per model

In [4]:
overview = raw[raw.exhibit == EXHIBIT].groupby('model').agg(
    answers=('run', 'size'),
    no_name=('failed', 'sum'),
    out_tokens_avg=('usage', lambda u: round(sum(x['out'] for x in u) / len(u))),
    seconds_avg=('seconds', 'mean'),
    cost_total=('cost', 'sum'),
).round({'seconds_avg': 1, 'cost_total': 4}).sort_values('cost_total', ascending=False)
print(f"total cost: ${overview.cost_total.sum():.2f}")
overview

total cost: $5.73


,answers,no_name,out_tokens_avg,seconds_avg,cost_total
model,,,,,
Muse Wonder,50,0,1256,13.5,1.2805
Gemini 4 Carbon,50,0,2119,33.6,1.0926
Qwen3.8 Max Prime,50,0,1344,17.5,0.8705
Claude Opus 5.5,50,0,524,7.0,0.6022
Gemini 4 Hydrogen,50,0,1382,9.4,0.5431
Mistral Large 4,50,0,4209,70.4,0.4454
Grok 4.7,50,0,812,17.5,0.2950
GPT-6.1 Sol,50,0,428,15.6,0.2431
Kimi K3.1 Pro,50,0,339,15.2,0.1867


## Every name, by model

In [5]:
wide = df.sort_values('run').groupby('model').title.apply(list).apply(pd.Series).T
wide.index = wide.index + 1
wide[models]

model,Claude Opus 5.5,Claude Sonnet 5.5,GLM 5.3 Prime,GPT-6 Luna,GPT-6.1 Sol,Gemini 4 Carbon,Gemini 4 Hydrogen,Grok 4.7,Kimi K3.1 Pro,Mistral Large 4,Muse Wonder,Qwen3.8 Max Prime
1,The Borrowed Lantern,The Lantern Moth,The Lanternfish,The Lantern Moth,The Unlatched Lantern,The Weaver's Prism,The Cartographer's Moth,The Mended Net,The Listening Fox,The Midnight Librarian,The Lantern-Bearing Moth,The Quiet Mender
2,The Listening Lantern,The Lantern Moth,The Lantern-Owl,The Questioning Compass,The Threadkeeper Fox,The Glass Cartographer,The Lantern Weaver,The Lantern Fox,The Listening Lantern,The Moth Spiral,The Lantern Fox,The Seamwright
3,The Kite of Questions,The Lantern Moth of Questions,The Lantern-Bearing Fox,The Star-Mending Moth,The Mapmaker’s Moth,The Magpie's Compass,The Paper Moth,The Ember Threadspinner,The Lantern Listener,The Mender's Seam,The Moon's Librarian,The Threadbare Fox
4,The Listening Heron,The Lantern Fox of Questions,The Lantern-Bearing Fox,The Question Moth,The Unfinished Lantern,The Bookbinder's Needle,The Whispering Astrolabe,The Threadmender,The Cartographer’s Moth,The Star Seamstress,The Star-Mending Fox,The Half-Remembered Door
5,The Lantern Moth,The Lantern Moth of Questions,The Cartographer's Lantern,The Cartographer Moth,The Unfinished Lantern,The Cartographer's Kite,The Lantern Moth,The Threadmender,The Listening Fox,The Moth's Map,The Moon's Librarian,The Tending Moth
6,The Lantern Fox,The Lantern Moth,The Cartographer's Lantern,Moth of Elsewhere,The Unfinished Hare,The Cartographer's Moth,The Star-Mender's Loom,The Lantern Fox,The Moth Cartographer,The Paper Boat,The Hedgehog Astronomer,The Mending Needle
7,The Kite of Questions,The Lantern Moth of Questions,The Lantern-Eared Fox,The Lantern Moth,The Unfinished Lantern,The Cartographer's Moth,The Paper Astrolabe,The Lantern Moth,The Lantern of Perhaps,The Star-Mender,The Wonder Weaver,The Cartographer's Moth
8,The Asking Heron,The Curious Lantern-Moth,The Lantern-Octopus,The Kite of Elsewhere,The Unfinished Fox,The Cartographer's Moth,The Star-Mender's Needle,The Ember Librarian,The Moth Cartographer,The Cat's Cradle,The Star-Gatherer's Cat,The Patient Needle
9,The Moth Who Asked,The Lantern Moth,The Lantern-Eared Fox,The Lantern Fox,The Unfinished Lantern,The Weaver's Shuttle,The Paper Moth,The Margin Fox,The Listening Compass,The Little Mapmaker,The Lantern Librarian,The Kite-Mender
10,The Lantern Hare,The Lantern Moth,The Kite That Escaped,The Wonder Kite,The Lantern Snail,The Glass Cartographer,The Loom of Echoes,The Lantern Scribe,The Cartographer’s Moth,The Map of Almost,The Listening Fox,The Cupped Whisper


## How often does each model repeat itself?

- **distinct names**: how many different names (after normalizing) out of its answers.
- **top name**: its most common name and how many times it gave it.
- **distinct figures**: how many different things (moth, fox, lantern…) its constellations are.
- **figure entropy**: spread of figures in bits; 0 means always the same figure, log2(10) ≈ 3.3 means all different.
- **name overlap**: average word overlap (Jaccard) between two of its names; 0 = no shared words, 1 = identical.

In [6]:
def entropy(values):
    counts = Counter(values); n = sum(counts.values())
    return -sum(c / n * math.log2(c / n) for c in counts.values())

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 0

def mean_overlap(word_lists):
    pairs = list(combinations(word_lists, 2))
    return sum(jaccard(a, b) for a, b in pairs) / len(pairs) if pairs else float('nan')

per_model = []
for m, g in df.groupby('model'):
    top, top_n = Counter(g.name).most_common(1)[0]
    per_model.append({
        'model': m, 'answers': len(g),
        'distinct names': g.name.nunique(), 'top name': top, 'top name ×': top_n,
        'distinct figures': g.figure.nunique(), 'top figure': Counter(g.figure).most_common(1)[0][0],
        'figure entropy': round(entropy(g.figure), 2),
        'name overlap': round(mean_overlap(g.words.tolist()), 2),
    })
repeat = pd.DataFrame(per_model).set_index('model').sort_values('name overlap', ascending=False)
repeat.style.bar(subset=['name overlap'], color='#f4a261', vmin=0, vmax=1) \
    .bar(subset=['distinct names', 'distinct figures'], color='#8ecae6', vmin=0, vmax=repeat.answers.max())

,answers,distinct names,top name,top name ×,distinct figures,top figure,figure entropy,name overlap
model,,,,,,,,
Claude Sonnet 5.5,50,6,lantern moth,31,3,moth,0.720000,0.690000
GLM 5.3 Prime,50,21,lantern bearing fox,18,11,fox,2.070000,0.350000
GPT-6.1 Sol,50,19,unfinished lantern,17,12,lantern,2.700000,0.270000
Kimi K3.1 Pro,50,22,listening lantern,14,10,lantern,2.610000,0.230000
GPT-6 Luna,50,24,question moth,14,11,moth,2.040000,0.220000
Claude Opus 5.5,50,27,lantern fox,12,10,fox,2.840000,0.160000
Gemini 4 Hydrogen,50,23,cartographer moth,9,13,moth,2.690000,0.150000
Gemini 4 Carbon,50,23,cartographer moth,14,15,moth,3.350000,0.140000
Grok 4.7,50,33,lantern wren,7,25,wren,4.170000,0.110000


## How much do the models repeat each other?

Word overlap between two answers from **different** models, averaged, next to the within-model overlap above. If the across-model number is close to the within-model one, the models are drawing from the same well.

In [7]:
across = [jaccard(a, b) for (ma, a), (mb, b) in combinations(zip(df.model, df.words), 2) if ma != mb]
within = [jaccard(a, b) for (ma, a), (mb, b) in combinations(zip(df.model, df.words), 2) if ma == mb]
print(f'average name overlap within a model:  {sum(within) / len(within):.2f}')
print(f'average name overlap across models:   {sum(across) / len(across):.2f}')
print(f'distinct names overall: {df.name.nunique()} of {len(df)}')

shared = df.groupby('name').agg(times=('model', 'size'), models=('model', lambda s: ', '.join(sorted(set(s)))))
shared['by how many models'] = df.groupby('name').model.nunique()
print('\nNames that more than one model came up with:')
shared[shared['by how many models'] > 1].sort_values(['by how many models', 'times'], ascending=False)

average name overlap within a model:  0.21


average name overlap across models:   0.09
distinct names overall: 259 of 600

Names that more than one model came up with:


,times,models,by how many models
name,,,
lantern moth,57,"Claude Opus 5.5, Claude Sonnet 5.5, GLM 5.3 Prime, GPT-6 Luna, GPT-6.1 Sol, ...",9
cartographer moth,34,"GLM 5.3 Prime, GPT-6 Luna, Gemini 4 Carbon, Gemini 4 Hydrogen, Kimi K3.1 Pro...",6
lantern fox,19,"Claude Opus 5.5, GLM 5.3 Prime, GPT-6 Luna, Gemini 4 Hydrogen, Grok 4.7, Mus...",6
listening fox,10,"Claude Opus 5.5, Grok 4.7, Kimi K3.1 Pro, Muse Wonder",4
listening loom,7,"GPT-6 Luna, Kimi K3.1 Pro, Muse Wonder, Qwen3.8 Max Prime",4
listening lantern,17,"Claude Opus 5.5, Grok 4.7, Kimi K3.1 Pro",3
moth cartographer,14,"GPT-6.1 Sol, Kimi K3.1 Pro, Qwen3.8 Max Prime",3
mender needle,7,"Gemini 4 Carbon, Mistral Large 4, Qwen3.8 Max Prime",3
lantern mender,6,"GPT-6 Luna, GPT-6.1 Sol, Grok 4.7",3


## The words everyone reaches for

Share of answers whose name contains each word, overall and per model (top 25 words).

In [8]:
def shade(v, vmax):
    a = 0 if pd.isna(v) or vmax == 0 else v / vmax
    return f'background-color: rgba(231, 111, 81, {a:.2f})'

has = df[['model']].copy()
top_words = [w for w, _ in Counter(w for ws in df.words for w in set(ws)).most_common(25)]
for w in top_words:
    has[w] = df.words.map(lambda ws, w=w: w in ws)
word_share = has.groupby('model')[top_words].mean().loc[models]
word_share.loc['ALL MODELS'] = has[top_words].mean()
(word_share * 100).round().astype(int).style.map(lambda v: shade(v, 100)).format('{}%')

,lantern,moth,fox,cartographer,listening,mender,unfinished,question,kite,questions,star,bearing,needle,loom,librarian,weaver,paper,keeper,heron,wren,cat,mending,lamplighter,echoes,astrolabe
model,,,,,,,,,,,,,,,,,,,,,,,,,
Claude Opus 5.5,46%,14%,26%,0%,20%,0%,2%,2%,8%,8%,0%,0%,0%,0%,0%,0%,0%,0%,20%,0%,0%,0%,14%,0%,0%
Claude Sonnet 5.5,98%,84%,14%,0%,0%,0%,0%,2%,0%,32%,0%,0%,0%,0%,0%,0%,0%,2%,0%,0%,0%,0%,2%,0%,0%
GLM 5.3 Prime,82%,6%,62%,20%,0%,0%,0%,0%,6%,0%,0%,36%,0%,0%,0%,0%,0%,8%,0%,0%,0%,0%,0%,0%,0%
GPT-6 Luna,16%,62%,4%,10%,18%,2%,0%,34%,14%,0%,6%,0%,0%,6%,0%,0%,0%,0%,0%,0%,0%,2%,0%,0%,0%
GPT-6.1 Sol,50%,24%,24%,10%,0%,4%,54%,0%,0%,0%,0%,0%,2%,2%,2%,2%,0%,0%,0%,2%,0%,0%,0%,0%,0%
Gemini 4 Carbon,16%,30%,4%,40%,0%,4%,0%,0%,10%,0%,0%,0%,6%,0%,0%,10%,10%,0%,0%,0%,0%,0%,0%,2%,4%
Gemini 4 Hydrogen,28%,46%,2%,18%,0%,8%,0%,0%,0%,0%,10%,0%,6%,16%,0%,6%,10%,0%,2%,2%,0%,0%,0%,12%,10%
Grok 4.7,48%,6%,14%,2%,4%,6%,0%,2%,0%,0%,0%,0%,2%,0%,4%,4%,0%,2%,2%,18%,0%,0%,0%,0%,0%
Kimi K3.1 Pro,42%,26%,16%,28%,50%,4%,2%,2%,0%,2%,4%,0%,0%,4%,0%,0%,0%,0%,0%,0%,0%,2%,0%,0%,0%


## What the constellations are (the figure), per model

In [9]:
top_figs = [f for f, _ in Counter(df.figure).most_common(15)]
figs = pd.crosstab(df.model, df.figure)[top_figs].loc[models]
figs.loc['ALL MODELS'] = figs.sum()
figs.style.map(lambda v: shade(v, figs.drop('ALL MODELS').values.max()))

figure,moth,fox,lantern,cartographer,kite,loom,mender,needle,librarian,heron,wren,weaver,cat,keeper,question
model,,,,,,,,,,,,,,,
Claude Opus 5.5,7,13,6,0,4,0,0,0,0,10,0,0,0,0,1
Claude Sonnet 5.5,42,7,0,0,0,0,0,0,0,0,0,0,0,0,1
GLM 5.3 Prime,3,31,6,1,3,0,0,0,0,0,0,0,0,0,0
GPT-6 Luna,31,2,1,0,7,3,1,0,0,0,0,0,0,0,0
GPT-6.1 Sol,6,12,18,5,0,1,2,0,1,0,1,1,0,0,0
Gemini 4 Carbon,15,0,7,2,5,0,0,3,0,0,0,0,0,0,0
Gemini 4 Hydrogen,23,1,1,0,0,8,0,3,0,1,1,3,0,0,0
Grok 4.7,3,7,2,1,0,0,3,1,2,1,9,2,0,1,1
Kimi K3.1 Pro,6,8,18,10,0,2,2,0,0,0,0,0,0,0,0


## The descriptions: most common words

In [10]:
desc = Counter(w for ws in df.desc_words for w in set(ws))
pd.DataFrame(desc.most_common(30), columns=['word', 'answers']).assign(
    share=lambda d: (d.answers / len(df) * 100).round().astype(int).astype(str) + '%'
).set_index('word').T

word,stars,dark,sky,star,night,small,light,question,lantern,moth,...,tail,seven,home,bright,toward,quiet,long,find,travelers,stray
answers,364,299,264,246,216,205,174,156,151,142,...,86,86,85,83,78,78,76,75,73,72
share,61%,50%,44%,41%,36%,34%,29%,26%,25%,24%,...,14%,14%,14%,14%,13%,13%,13%,12%,12%,12%


## Compared with the gallery: each model's actual piece

In [11]:
gallery = []
for f in (ROOT.parents[1] / 'exhibits' / EXHIBIT / 'entries').glob('*/meta.json'):
    m = json.loads(f.read_text())
    if m.get('title'):
        gallery.append({'model': m['artist'], 'gallery piece': m['title']})
g = pd.DataFrame(gallery).set_index('model')
g['among its answers?'] = [normalize(t) in set(df[df.model == m].name) for m, t in g['gallery piece'].items()]
g['its most common name here'] = repeat['top name'].reindex(g.index)
g.loc[[m for m in models if m in g.index]]

,gallery piece,among its answers?,its most common name here
model,,,
Claude Opus 5.5,The Lamplighter,False,lantern fox
Claude Sonnet 5.5,The Moth Who Borrowed the Moon,False,lantern moth
GLM 5.3 Prime,The Thousandth Crane,False,lantern bearing fox
GPT-6 Luna,The Wish-Moth,False,question moth
GPT-6.1 Sol,The Lantern Heron,False,unfinished lantern
Gemini 4 Carbon,The Spindle Moth,False,cartographer moth
Gemini 4 Hydrogen,The Lantern Moth,True,cartographer moth
Grok 4.7,The Thousandth Crane,False,lantern wren
Kimi K3.1 Pro,The Thimble Moth,False,listening lantern


## Is 10 answers per model enough?

Distinct names among each model's first *k* answers, as *k* grows. If a model's count stops climbing, more answers won't show much new; if it keeps climbing in step with *k*, the model rarely repeats and more answers would keep finding new names.

In [12]:
ks = [k for k in (3, 5, 10, 20, 30, 50) if k <= df.groupby('model').size().min()]
curve = pd.DataFrame({f'first {k}': df[df.run <= k].groupby('model').name.nunique() for k in ks}).loc[models]
curve.loc['ALL MODELS'] = [df[df.run <= k].name.nunique() for k in ks]
curve

,first 3,first 5,first 10,first 20,first 30,first 50
model,,,,,,
Claude Opus 5.5,3,5,9,14,18,27
Claude Sonnet 5.5,2,3,4,5,5,6
GLM 5.3 Prime,3,4,7,11,15,21
GPT-6 Luna,3,5,9,11,15,24
GPT-6.1 Sol,3,4,7,9,13,19
Gemini 4 Carbon,3,5,7,15,18,23
Gemini 4 Hydrogen,3,5,9,13,19,23
Grok 4.7,3,4,8,14,22,33
Kimi K3.1 Pro,3,4,7,12,18,22


## Failures

Answers with no name in them (the reply is kept in the `.failed.json` file).

In [13]:
failed[['model', 'run', 'stop']].assign(reply_start=failed.reply.str[:120]) if len(failed) else 'none'

'none'